## Matrix Multiplication with S3 Persistence

A compact end-to-end example that computes a matrix product and stores the result in AWS S3 using LAILA.

### What this notebook does
- Generate two random matrices and compute `C = A @ B`
- Wrap `C` as a LAILA entry (`laila.constant`) to get a global ID
- Persist the entry to S3 via `S3Pool` (`memorize`)
- Retrieve the entry from S3 (`remember`)
- Remove the stored object (`forget`) for cleanup

### Credentials
This notebook reads AWS credentials from:
`laila.args`


In [1]:
%load_ext autoreload
%autoreload 2

In [3]:
import laila
import time
import numpy as np
import matplotlib.pyplot as plt

In [4]:
from laila.pool import RedisPool

redis_pool = RedisPool()

In [5]:
laila.add_pool(redis_pool, pool_nickname="my_redis_pool")

In [6]:
import torch

# Create different Python data types
sample_int = 42
sample_float = 3.14159
sample_str = "hello laila"
sample_bool = True
sample_none = None
sample_list = [1, 2, 3, "four"]
sample_tuple = ("alpha", "beta", "gamma")
sample_set = {"x", "y", "z"}
sample_dict = {"name": "laila", "version": 1, "active": True}
sample_numpy = np.array([[1.0, 2.0], [3.0, 4.0]])
sample_torch_tensor = torch.tensor([[10, 20], [30, 40]], dtype=torch.float32)

# Wrap each value as a LAILA constant to assign global IDs
wrapped_constants = {
    "int": laila.constant(data=sample_int),
    "float": laila.constant(data=sample_float),
    "str": laila.constant(data=sample_str),
    "bool": laila.constant(data=sample_bool),
    "none": laila.constant(data=sample_none),
    "list": laila.constant(data=sample_list),
    "tuple": laila.constant(data=sample_tuple),
    "set": laila.constant(data=sample_set),
    "dict": laila.constant(data=sample_dict),
    "numpy": laila.constant(data=sample_numpy),
    "torch_tensor": laila.constant(data=sample_torch_tensor),
}

# Quick check: print each constant's global_id
for dtype, wrapped in wrapped_constants.items():
    print(f"{dtype:>12} -> {wrapped.global_id}")

         int -> LAILA:ENTRY:GLOBAL_ID:1edae535-acb7-4130-8d60-e836ad4b40b3
       float -> LAILA:ENTRY:GLOBAL_ID:158e020b-ccde-4d5b-8236-283e602dd9ad
         str -> LAILA:ENTRY:GLOBAL_ID:7d714fef-6184-41ea-821c-a23324b2b89d
        bool -> LAILA:ENTRY:GLOBAL_ID:4a998ff1-19c5-4981-a6a8-684c6928b96a
        none -> LAILA:ENTRY:GLOBAL_ID:f66b00b1-910f-4570-99de-56994d6bc5c5
        list -> LAILA:ENTRY:GLOBAL_ID:8034f048-88a7-4e94-b4f1-c26dd238f852
       tuple -> LAILA:ENTRY:GLOBAL_ID:c7ee886c-c3dc-4855-9aa4-7b44474a4c87
         set -> LAILA:ENTRY:GLOBAL_ID:7378dd22-f33a-4b0f-a020-60d43c7b19bd
        dict -> LAILA:ENTRY:GLOBAL_ID:4092653e-7bb5-4bc5-b1c0-127e41979e17
       numpy -> LAILA:ENTRY:GLOBAL_ID:2e072630-30e4-4f2b-817a-97a832f2b47a
torch_tensor -> LAILA:ENTRY:GLOBAL_ID:5c46f8fa-f28b-4ba3-9495-fd29efcd50da
